# Warhammer 40K Miniature Detector - YOLO11x Training (Run 2)

Train the **best available YOLO model** (YOLO11x - 57M parameters) on your annotated dataset.

**Dataset**: 593 images, 2,656 bounding boxes, 15 factions

## Setup
1. **Runtime > Change runtime type > T4 GPU** (free tier)
2. Upload `yolo_dataset.zip` when prompted (201 MB)
3. Run all cells
4. Training takes ~2-3 hours on T4 GPU

## Expected Results
- Previous model (YOLOv8n, 280 images, 8 classes): 63.2% mAP50
- Target (YOLO11x, 593 images, 15 classes): 70-80% mAP50

In [ ]:
# Install YOLOv8/YOLO11
!pip install ultralytics -q
print("Ultralytics installed!")

In [ ]:
# Check GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Upload your dataset
from google.colab import files
print("Upload yolo_dataset.zip (201 MB)")
print("You can find this at: photoanalyzer/backend/yolo_dataset.zip")
uploaded = files.upload()

In [ ]:
# Extract dataset
!unzip -q yolo_dataset.zip -d /content/
!ls /content/yolo_dataset/

import os
train_count = len(os.listdir('/content/yolo_dataset/images/train'))
val_count = len(os.listdir('/content/yolo_dataset/images/val'))
print(f"\nTrain images: {train_count}")
print(f"Val images: {val_count}")
print(f"Total: {train_count + val_count}")

In [ ]:
# Update data.yaml path for Colab
yaml_content = '''# YOLO Dataset Configuration
path: /content/yolo_dataset
train: images/train
val: images/val

# Classes
nc: 15
names: ["adeptus_mechanicus", "chaos_knights", "chaos_space_marines", "custodes", "death_guard", "deathwatch", "eldar", "genestealer_cult", "grey_knights", "imperial_guard", "necrons", "orks", "space_marines", "thousand_sons", "tyranids"]
'''

with open('/content/yolo_dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml updated for Colab!")
!cat /content/yolo_dataset/data.yaml

## Training with YOLO11x

YOLO11x is the latest and most accurate YOLO model:
- 57M parameters (vs 3M for YOLOv8n)
- State-of-the-art accuracy
- ~2-3 hours training on T4 GPU
- 593 images / 15 classes (2x more data, 2x more classes than run 1)

In [ ]:
from ultralytics import YOLO

# Load YOLO11x - the best model
model = YOLO('yolo11x.pt')
print("YOLO11x loaded!")

In [ ]:
# Train with optimal settings for 593-image dataset
results = model.train(
    data='/content/yolo_dataset/data.yaml',
    
    # Training duration
    epochs=150,
    patience=20,  # Early stopping — more patience for 15 classes
    
    # Resolution
    imgsz=640,
    
    # Batch size (T4 has 15GB VRAM — 8 is safe for YOLO11x at 640)
    batch=8,
    
    # Augmentation (aggressive — we need it with 593 images)
    degrees=15,
    translate=0.15,
    scale=0.5,
    shear=5,
    flipud=0.1,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.15,
    
    # Regularization
    dropout=0.15,
    
    # Learning rate — slightly lower for stability with more classes
    lr0=0.008,
    lrf=0.01,
    warmup_epochs=5,
    
    # Output
    save=True,
    save_period=10,
    project='/content/runs',
    name='warhammer_yolo11x_r2',
    exist_ok=True,
    verbose=True,
    plots=True,
)

In [ ]:
# View training results
from IPython.display import Image, display
display(Image('/content/runs/warhammer_yolo11x_r2/results.png', width=800))

In [ ]:
# View confusion matrix
display(Image('/content/runs/warhammer_yolo11x_r2/confusion_matrix.png', width=600))

In [ ]:
# Validate and show final metrics
best_model = YOLO('/content/runs/warhammer_yolo11x_r2/weights/best.pt')
metrics = best_model.val(data='/content/yolo_dataset/data.yaml')

print("\n" + "="*50)
print("FINAL MODEL METRICS (Run 2)")
print("="*50)
print(f"mAP50:     {metrics.box.map50:.1%}")
print(f"mAP50-95:  {metrics.box.map:.1%}")
print(f"Precision: {metrics.box.mp:.1%}")
print(f"Recall:    {metrics.box.mr:.1%}")
print("="*50)
print(f"\nPrevious run: 63.2% mAP50 (YOLOv8n, 280 images, 8 classes)")
print(f"This run:     {metrics.box.map50:.1%} mAP50 (YOLO11x, 593 images, 15 classes)")

# Per-class results
print("\nPer-class mAP50:")
for i, name in enumerate(metrics.names.values()):
    ap = metrics.box.ap50[i]
    print(f"  {name:<25} {ap:.1%}")

In [ ]:
# Test on validation images
import os
val_images = os.listdir('/content/yolo_dataset/images/val/')[:6]

for img in val_images:
    results = best_model.predict(
        f'/content/yolo_dataset/images/val/{img}',
        save=True,
        project='/content/predictions',
        conf=0.25
    )
    print(f"Processed {img}")

In [ ]:
# Show predictions
import glob
pred_images = sorted(glob.glob('/content/predictions/predict/*.jpg'))[:6]
for img in pred_images:
    print(f"\n{os.path.basename(img)}")
    display(Image(img, width=500))

## Download Trained Model

Download the best model weights to use in your application.

In [ ]:
# Download the trained model
from google.colab import files

print("Downloading best.pt (~110MB)...")
files.download('/content/runs/warhammer_yolo11x_r2/weights/best.pt')

In [ ]:
# Optional: Download all training artifacts
!zip -r /content/training_results.zip /content/runs/warhammer_yolo11x_r2/
files.download('/content/training_results.zip')

## Using the Model

After downloading, use the model like this:

```python
from ultralytics import YOLO

# Load your trained model
model = YOLO('best.pt')

# Run inference
results = model.predict('image.jpg', conf=0.25)

# Get detections
for r in results:
    for box in r.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        name = model.names[cls]
        print(f"{name}: {conf:.1%}")
```